# Gemini 大文件理解 API 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台，使用 **OpenAI Python SDK** 上传大文件并调用 **Gemini** 模型进行多模态理解（图片理解、视频理解等）。

## 功能简介

Vertex AI 通过 HTTP 直接传入文件进行理解时，存在 **15MB** 的硬性限制。对于大于 15MB 的文件（如长视频、高分辨率图像），官方要求必须使用 Google Cloud Storage (GCS) 链接。

七牛 AIToken 平台提供了 **文件上传接口**，简化了这一流程：
- 您只需提供文件的公开 URL，系统将自动把文件转存至 GCS
- 上传完成后获得 `file_id`（格式为 `qfile-xxx`），即可在 Chat Completions 接口中直接使用
- 最大支持 **2GB** 文件

## 调用流程

1. **提交文件上传任务** — 调用 `POST /v1/files`，传入 `source_url`
2. **轮询等待文件就绪** — 调用 `GET /v1/files/{file_id}`，直到状态变为 `ready`
3. **调用 Chat Completions** — 在消息中使用 `file_id` 引用已上传的文件

**API 端点**：`https://api.qnaigc.com/v1`  
**适用模型**：`gemini-3.0-pro-preview`、`gemini-3.1-pro-preview`、`gemini-2.0-flash` 等 Gemini 系列模型

## 1. 环境准备

安装 `openai` 和 `requests` 依赖包，并设置 API Key。

In [ ]:
# 安装依赖
%pip install openai requests -q

In [ ]:
import os
import time
import requests
from openai import OpenAI

# 七牛 AIToken 平台地址
BASE_URL = "https://api.qnaigc.com/v1"

# 从环境变量读取 API Key
API_KEY = os.environ.get("QINIU_API_KEY", "<your-api-key>")

# 初始化 OpenAI 客户端，指向七牛 AIToken 平台
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
)

# 使用的模型
MODEL = "gemini-3.0-pro-preview"

## 2. 文件上传工具函数

封装文件上传和状态轮询逻辑。文件上传是**异步**操作：
1. 调用 `POST /v1/files` 提交任务，立即返回 `file_id`
2. 轮询 `GET /v1/files/{file_id}`，等待状态变为 `ready`

文件状态说明：
- `pending`：等待处理
- `uploading`：正在上传到目标存储
- `ready`：上传完成，可用于推理
- `failed`：上传失败
- `expired`：文件已过期

In [ ]:
def create_file(source_url: str, model: str = MODEL) -> dict:
    """创建文件上传任务，返回文件信息（包含 file_id）"""
    resp = requests.post(
        f"{BASE_URL}/files",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": model,
            "source_url": source_url,
        },
    )
    resp.raise_for_status()
    return resp.json()


def get_file_status(file_id: str) -> dict:
    """查询文件处理状态"""
    resp = requests.get(
        f"{BASE_URL}/files/{file_id}",
        headers={"Authorization": f"Bearer {API_KEY}"},
    )
    resp.raise_for_status()
    return resp.json()


def wait_for_file_ready(file_id: str, timeout: int = 300, interval: int = 3) -> dict:
    """轮询等待文件处理完成，返回文件信息"""
    deadline = time.time() + timeout
    while time.time() < deadline:
        file_info = get_file_status(file_id)
        status = file_info.get("status")
        print(f"  文件状态: {status}")

        if status == "ready":
            return file_info
        elif status == "failed":
            error = file_info.get("error", {})
            raise RuntimeError(f"文件处理失败: {error.get('message', 'unknown error')}")
        elif status == "expired":
            raise RuntimeError("文件已过期")

        # pending 或 uploading 状态，继续等待
        time.sleep(interval)

    raise TimeoutError(f"等待文件就绪超时（{timeout}s）")


def upload_and_wait(source_url: str, model: str = MODEL) -> str:
    """上传文件并等待就绪，返回 file_id"""
    # 第一步：创建文件上传任务
    print(f"正在提交文件上传任务...")
    print(f"  source_url: {source_url}")
    file_info = create_file(source_url, model)
    file_id = file_info["id"]
    print(f"  file_id: {file_id}")
    print(f"  status: {file_info['status']}")

    # 第二步：轮询等待文件就绪
    print(f"\n等待文件处理完成...")
    file_info = wait_for_file_ready(file_id)
    print(f"文件已就绪!")
    print(f"  file_id:      {file_info['id']}")
    print(f"  file_name:    {file_info.get('file_name', 'N/A')}")
    print(f"  file_size:    {file_info.get('file_size', 'N/A')} bytes")
    print(f"  content_type: {file_info.get('content_type', 'N/A')}")

    return file_id

## 3. 示例一：图片理解

上传一张图片，然后使用 Gemini 模型进行图片内容理解。

在 Chat Completions 接口中，通过 `file` 类型的 content part 传入 `file_id`，平台会自动将其替换为 GCS 地址。

In [ ]:
# 上传图片
image_url = "https://aitoken-public.qnaigc.com/example/generate-video/running-man.jpg"
image_file_id = upload_and_wait(image_url)

In [ ]:
# 使用 file 类型调用 Chat Completions 进行图片理解
response = client.chat.completions.create(
    model=MODEL,
    stream=False,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "这张图片里有什么？请详细描述。",
                },
                {
                    "type": "file",
                    "file": {
                        "file_id": image_file_id,
                    },
                },
            ],
        }
    ],
)

print("=== 模型回复 ===")
print(response.choices[0].message.content)

## 4. 示例二：视频理解

上传一段视频，然后使用 Gemini 模型进行视频内容理解。

视频文件通常超过 15MB，必须通过文件上传接口上传后使用。

In [ ]:
# 上传视频
video_url = "https://aitoken-public.qnaigc.com/example/generate-video/the-little-dog-is-running-on-the-lawn.mp4"
video_file_id = upload_and_wait(video_url)

In [ ]:
# 使用 file 类型调用 Chat Completions 进行视频理解
response = client.chat.completions.create(
    model=MODEL,
    stream=False,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "这段视频里发生了什么？请详细描述。",
                },
                {
                    "type": "file",
                    "file": {
                        "file_id": video_file_id,
                        "format": "video/mp4",
                    },
                },
            ],
        }
    ],
)

print("=== 模型回复 ===")
print(response.choices[0].message.content)

## 5. 替代方式：通过 image_url 引用 file_id

除了使用 `file` 类型，也可以通过 `image_url` 类型传入 `file_id`。只需将 `url` 字段设置为 `qfile-xxx` 格式的 file_id 即可。

两种方式等价，平台都会自动将 `qfile-xxx` 替换为实际的 GCS 文件地址。

In [ ]:
# 通过 image_url 方式引用已上传的图片
response = client.chat.completions.create(
    model=MODEL,
    stream=False,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Describe this image in English.",
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_file_id,  # 直接传入 qfile-xxx 格式的 file_id
                    },
                },
            ],
        }
    ],
)

print("=== 模型回复 ===")
print(response.choices[0].message.content)

## 6. 参数说明

### 文件上传接口 `POST /v1/files`

| 参数 | 类型 | 必填 | 说明 |
|------|------|------|------|
| `model` | string | 是 | 目标模型 ID，如 `gemini-3.0-pro-preview`。同系列模型创建的 file_id 可以共用 |
| `source_url` | string | 是 | 公开可访问的文件 URL（HTTP/HTTPS） |
| `expires_in` | integer | 否 | 过期时间（秒），范围 3600~604800，默认 172800（48小时） |

### Chat Completions 中引用文件

**方式一：file 类型**
```json
{
    "type": "file",
    "file": {
        "file_id": "qfile-xxx",
        "format": "video/mp4"
    }
}
```

**方式二：image_url 类型**
```json
{
    "type": "image_url",
    "image_url": {
        "url": "qfile-xxx"
    }
}
```

## 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [OpenAI Python SDK 文档](https://github.com/openai/openai-python)
- [七牛 API 调用示例文档](https://apidocs.qnaigc.com/417774894e0)